# GP2 GBA1 MAVE Project
## Extract variant carriers and perform annotation using GP2 WGS data

### Set Up

In [ ]:
## Use the os package to interact with the environment
import os

## Bring in Pandas for Dataframe functionality
import pandas as pd

import subprocess

## Numpy for basics
import numpy as np

## Use pathlib for file path manipulation
import pathlib

## Use StringIO for working with file contents
from io import StringIO

## Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

## Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

## Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

## BigQuery for querying data
from google.cloud import bigquery

## Import Sys
import sys as sys

#### Install plink

In [ ]:
%%capture
%%bash

# Install plink 1.9
mkdir -p ~/tools
cd ~/tools/
if test -e ~/tools/plink; then
    echo "Plink is already installed"
else
    echo "Plink is not installed"
    cd ~/tools/

    wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

In [ ]:
%%bash

# chmod plink 1.9 to make sure you have permission to run the program
chmod u+x ~/tools/plink1.9

In [ ]:
%%capture
%%bash

# Install plink 2.0
cd ~/tools/
if test -e ~/tools/plink2; then
    echo "Plink2 is already installed"
else
    echo "Plink2 is not installed"
    cd ~/tools/

    wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

    unzip -o plink2_linux_x86_64_latest.zip
fi

In [ ]:
%%bash

# chmod plink 2 to make sure you have permission to run the program
chmod u+x ~/tools/plink2

#### Install bcftools

In [ ]:
%%capture
%%bash 

#install bcftools
cd /home/jupyter/tools/

if test -e /home/jupyter/tools/bcftools; then
    echo "bcftools is already installed in /home/jupyter/tools/"
else
    echo -e "Downloading bcftools \n    -------"
    git clone --recurse-submodules https://github.com/samtools/htslib.git
    git clone https://github.com/samtools/bcftools.git
    cd bcftools
    make
    echo -e "\n bcftools downloaded and unzipped in /home/jupyter/tools \n "

fi

In [ ]:
%%bash

# chmod bcftools to make sure you have permission to run the program
chmod +x /home/jupyter/tools/bcftools

#### Install annovar

In [ ]:
%%capture
%%bash

# Install ANNOVAR: We are adding the download link after registration on the annovar website
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e ~/tools/annovar ; then
    echo "annovar is already installed in /home/jupyter/workspace/ws_files/fangz_workdir/annovar"
else
    echo "annovar is not installed"
    cd ~/tools

    wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

    tar xvfz annovar.latest.tar.gz

fi

In [ ]:
%%capture
%%bash

# Install ANNOVAR: Download resources for annotation

cd ~/tools/annovar
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20240917 humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_genome humandb/

### Create working directory and set paths

In [ ]:
# Create a folder on your workspace
print("Making a working directory")
!mkdir -p /home/jupyter/workspace/ws_files/GBA1_MAVE
!mkdir -p /home/jupyter/workspace/ws_files/GBA1_MAVE/bed_files
!mkdir -p /home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files
!mkdir -p /home/jupyter/workspace/ws_files/GBA1_MAVE/results

workdir="/home/jupyter/workspace/ws_files/GBA1_MAVE"

### Create and edit bed file with genes of interest and coordinates

#### Create a .txt file with the genetic variant information (include all variants you are interested in)
Use the following format: chr:pos:ref:alt 

In [ ]:
#Convert your .txt file into a .bed file
input_file = "/home/jupyter/workspace/ws_files/GBA1_MAVE/bed_files/GBA1_MAVE.txt"
output_file = "/home/jupyter/workspace/ws_files/GBA1_MAVE/bed_files/GBA1_MAVE.bed"

with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        cleaned_line = "\t".join(line.strip().split())  # Replace spaces with actual tabs
        outfile.write(cleaned_line + "\n")

print("Conversion complete! File saved as:", output_file)

In [ ]:
#Read in the bed file
bed = pd.read_csv('/home/jupyter/workspace/ws_files/GBA1_MAVE/bed_files/GBA1_MAVE.bed', sep='\t',header=None,names=['chr','start_bp','stop_bp','gene'])
bed

### Extract variant carriers from WGS data

In [ ]:
# define all the variables
INPUT_PLINK_DIR="/home/jupyter/workspace/gp2_tier2_eu_release12/wgs/dragen_joint_calling/plink"
PLINK2_PATH="/home/jupyter/tools/plink2"
TEMP_DIR="/home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files"
Ancestry="EUR"

VARLIST="/home/jupyter/workspace/ws_files/GBA1_MAVE/bed_files/GBA1_MAVE.txt"
chrom = "chr1" 

import os, subprocess

os.makedirs(f"{TEMP_DIR}/{Ancestry}", exist_ok=True)

cmd = [
    PLINK2_PATH,
    "--pfile", f"{INPUT_PLINK_DIR}/{Ancestry}/chr1_{Ancestry}_release12",
    "--set-all-var-ids", "@:#:$r:$a",           
    "--new-id-max-allele-len", "50", "missing",   
    "--extract", VARLIST,                       
    "--mac", "1",
    "--make-pgen",
    "--out", f"{TEMP_DIR}/{Ancestry}/{chrom}"   
]
subprocess.run(cmd, check=True)

In [ ]:
%%bash

TEMP_DIR="/home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files"
Ancestry="EUR"

# strip the .pgen extension to get the dataset name
Dataset=$(ls ${TEMP_DIR}/${Ancestry}/*.pgen | sed 's/\.pgen$//')

# convert the single pfile set to VCF
~/tools/plink2 \
  --pfile ${Dataset} \
  --recode vcf \
  --out ${TEMP_DIR}/${Ancestry}

In [ ]:
%%bash
# bgzip the vcf
TEMP_DIR="/home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files"
Ancestry="EUR"

bgzip -c ${TEMP_DIR}/${Ancestry}.vcf > ${TEMP_DIR}/annovar_input_${Ancestry}.vcf.gz
tabix -p vcf ${TEMP_DIR}/annovar_input_${Ancestry}.vcf.gz

In [ ]:
%%bash
### Remove genotypes to make file smaller and easier to process
TEMP_DIR="/home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files"
Ancestry="EUR"

bcftools view -O z -G ${TEMP_DIR}/annovar_input_${Ancestry}.vcf.gz -o ${TEMP_DIR}/annovar_input_new_${Ancestry}.vcf.gz
bcftools index -t ${TEMP_DIR}/annovar_input_new_${Ancestry}.vcf.gz

In [ ]:
%%bash

TEMP_DIR="/home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files"

# Run ANNOVAR annotation
perl ~/tools/annovar/table_annovar.pl \
    /home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files/annovar_input_new_EUR.vcf.gz \
    /home/jupyter/tools/annovar/humandb/ \
    -buildver hg38 \
    -out /home/jupyter/workspace/ws_files/GBA1_MAVE/results/wgs_mac1_final_EUR.annovar \
    -remove \
    -protocol refGene,gnomad41_genome,clinvar_20240917 \
    -operation g,f,f \
    -nastring . \
    -polish \
    -vcfinput

In [ ]:
%%bash 
workdir="/home/jupyter/workspace/ws_files/GBA1_MAVE"

head ${workdir}/results/wgs_mac1_final_EUR.annovar.hg38_multianno.txt

In [ ]:
workdir="/home/jupyter/workspace/ws_files/GBA1_MAVE"
# Show columns of multianno.txt output file
anno=pd.read_csv(f'{workdir}/results/wgs_mac1_final_EUR.annovar.hg38_multianno.txt',sep='\t',dtype=str)

#find variant id col:
anno['Otherinfo6']

In [ ]:
anno.columns.tolist()

In [ ]:
# select the cols to keep
basic_cols= anno.columns.tolist()[0:10]
additional_cols_to_keep=['Otherinfo6',
                         'gnomad41_genome_fafmax_faf95_max',
                         'CLNDN',
                         'CLNSIG']
            
all_cols_to_keep= basic_cols+ additional_cols_to_keep
all_cols_to_keep

In [ ]:
# subset the columns
EUR_df=anno[all_cols_to_keep]

#rename col
EUR_df.rename({'Otherinfo6':'var_id'},axis=1,inplace=True)

EUR_df

In [ ]:
# Save the filtered output
EUR_df.to_csv(f"{workdir}/results/filtered_multianno_EUR.tsv", sep="\t", index=False)

# write out var_id to extract from plink files
EUR_df['var_id'].to_csv(f"{workdir}/results/EUR_var_to_extract.txt",index=False,header=False)


In [ ]:
%%bash
workdir="/home/jupyter/workspace/ws_files/GBA1_MAVE"
TEMP_DIR="/home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files"
Ancestry="EUR"

~/tools/plink2 --pfile ${TEMP_DIR}/${Ancestry}/chr1 \
               --extract ${workdir}/results/${Ancestry}_var_to_extract.txt \
               --recode A \
               --out ${TEMP_DIR}/${Ancestry}_geno
               

In [ ]:
TEMP_DIR="/home/jupyter/workspace/ws_files/GBA1_MAVE/temp_files"
Ancestry="EUR"

eur_var = pd.read_csv(f'{TEMP_DIR}/{Ancestry}_geno.raw', sep='\s+')
eur_var

In [ ]:
# transpose the dataframe to be row as variants and columns as samples
var_col=eur_var.columns[6:len(eur_var)]
d = eur_var.drop(columns=['FID','PAT','MAT','SEX','PHENOTYPE'])
sample=eur_var[['IID','PHENOTYPE']]

#Filtering rows where any value in var_col is ≤1 (either het or hom)
t=d[(d[var_col]<=1).any(axis=1)].T
t.columns = t.iloc[0]
t=t.iloc[1:]
t.reset_index(inplace=True)

t

In [ ]:
#strip the last '_${ref_allele}', so we can keep the same variant id as in annotation 
t['index'] = t['index'].str.rsplit('_', n=1).str[0]
t.rename({'index':'var_id'},axis=1,inplace=True)
t

In [ ]:
t['hom_carrier'] = t.apply(lambda row: row[row == 0].index.tolist() , axis=1)
t['het_carrier'] = t.apply(lambda row: row[row == 1].index.tolist() , axis=1)  

# store hom and het seperately to later explode the dataframe correctly
hom = t[['var_id','hom_carrier']]
het = t[['var_id','het_carrier']]

In [ ]:
# check hom as example what to expect
hom

In [ ]:
# split the carrier id from the list and each to a row
hom = hom.explode('hom_carrier', ignore_index=True)
hom

In [ ]:
# keep just the variants with carriers
hom= hom.loc[~hom['hom_carrier'].isnull()]
hom

In [ ]:
#merge with annotation
out_hom = pd.merge(hom,EUR_df, on='var_id',how='left')
out_hom['zygosity'] = 'hom'

#rename col
out_hom.rename({'hom_carrier':'carrier_id'},axis=1,inplace=True)
out_hom

In [ ]:
#repeat the same for het
het = het.explode('het_carrier', ignore_index=True)
het= het.loc[~het['het_carrier'].isnull()]
het

In [ ]:
#merge with annotation
out_het = pd.merge(het,EUR_df, on='var_id',how='left')
out_het['zygosity'] = 'het'

#rename col
out_het.rename({'het_carrier':'carrier_id'},axis=1,inplace=True)
out_het

In [ ]:
# get everything and write out
merged_df_EUR = pd.concat([out_het,out_hom],axis=0)

# Save the dataset
merged_df_EUR.to_csv(f"{workdir}/results/GBA1_MAVE_EUR.tsv",sep='\t',index=False)

### Extract clinical data for identified variant carriers

#### Read in GP2 clinical data 

In [ ]:
# Let's load the master key
master_key_df = pd.read_csv('/home/jupyter/workspace/gp2_tier2_eu_release12/clinical_data/master_key_release12_final_vwb.csv')
print(master_key_df.shape)

In [ ]:
print('Basic information:')
print()
master_key_df.info()

In [ ]:
# Extract only the necessary columns from the master_key_df
columns_to_extract = ['GP2ID', 'wgs_label', 'study_type', 'diagnosis', 'GP2_phenotype_for_qc', 'GP2_phenotype', 'GP2_PHENO', 'age_of_onset', 'age_at_diagnosis']  
master_key_subset = master_key_df[columns_to_extract]

print(master_key_subset)

In [ ]:
# Read in file with variant carriers
carrier = pd.read_csv('/home/jupyter/workspace/ws_files/GBA1_MAVE/results/GBA1_MAVE_EUR.csv', sep=',')
print(carrier)

In [ ]:
# Merge the dataframes on 'Sample_ID'
carrier_meta = pd.merge(carrier, master_key_subset, on='GP2ID', how='left')
print(carrier_meta)

In [ ]:
# write out data
carrier_meta.to_csv('/home/jupyter/workspace/ws_files/GBA1_MAVE/results/GBA1_MAVE_EUR-meta.csv', index=False)